(a): .group()
(b):'2026-05-06'
(c): ['2026-05-06', '2026-05-18']
(d): [('2026', '05', '06'), ('2026', '05', '18')]
(e): ['2026-05-06', '2026-05-18']
추가질문: c는 캡쳐그룹이 없어서 그냥 리스트의 형태로 전체 문자열을 반환한다
하지만 (\d{4}), (\d{2}), (\d{2})처럼 캡처 그룹이 있어서 전체 날짜가 아니라 각 그룹에 해당하는 연·월·일만 튜플 형태로 반환한다.e는 아예 비캡쳐 그룹이므로 그룹내용만 따로 뽑지는 않고 전체 매칭된 문자열을 반환핟다. 

'[T]!'
'[T]안녕[T] [T]세상[T]!'
'[T]안녕[T] [T]세상[T]!'
'수강생 <30>명, 조교 <3>명'
'수강생 <\x01>명, 조교 <\x01>명'

i) ((a)의 <.+>는 탐욕적 수량자라 첫 번째 <부터 마지막 >까지 한 번에 매칭하지만, (b)의 <.+?>는 게으른 수량자라 <b>, </b>, <i>, </i>처럼 각 태그를 가장 짧게 매칭하므로 결과가 다르다.
ii) (d)의 r"<\1>"에서는 \1이 1번 캡처 그룹인 숫자를 의미하지만, (e)의 "<\1>"에서는 \1이 원시 문자열이 아니어서 Python 문자열 안에서 \x01로 먼저 해석되므로 결과가 다르다.

In [16]:
import re
from collections import Counter

URL_RE = re.compile(r"https?://\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMAIL_RE = re.compile(r"\b[\w.-]+@[\w.-]+\.\w+\b")
PHONE_RE = re.compile(r"\b\d{2,4}-\d{3,4}-\d{4}\b")
MENTION_RE = re.compile(r"@\w+")
JAMO_RE = re.compile(r"[\u3131-\u3163]+")
SPACE_RE = re.compile(r"\s+")

def clean_post(post: str) -> str:
    post = URL_RE.sub(" ", post)
    post = HTML_TAG_RE.sub("", post)
    post = EMAIL_RE.sub("[이메일]", post)
    post = PHONE_RE.sub("[전화번호]", post)
    post = MENTION_RE.sub(" ", post)
    post = JAMO_RE.sub("", post)
    post = SPACE_RE.sub(" ", post).strip()
    return post

HASHTAG_RE = re.compile(r"#([가-힣a-zA-Z0-9]+)")

def extract_hashtags(post: str) -> list[str]:
    return HASHTAG_RE.findall(post)


from collections import Counter

def analyze_posts(posts: list[str]) -> dict:
    cleaned_posts = [clean_post(post) for post in posts]

    total_chars = sum(len(post) for post in cleaned_posts)

    hashtag_counter = Counter()
    masked_count = 0

    for post in posts:
        hashtag_counter.update(extract_hashtags(post))

        _, email_n = EMAIL_RE.subn("[이메일]", post)
        _, phone_n = PHONE_RE.subn("[전화번호]", post)
        masked_count += email_n + phone_n

    sorted_hashtags = dict(
        sorted(
            hashtag_counter.items(),
            key=lambda item: (-item[1], item[0])
        )
    )

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": round(total_chars / len(posts), 2) if posts else 0.0,
        "hashtag_counts": sorted_hashtags,
        "masked_count": masked_count,
    }

In [17]:
posts = [
    "오늘 파이썬 수업 진짜 재밌었음!! @prof_kim ㅎㅎㅎ",
    "자료: https://etl.snu.ac.kr/lec17",
    "lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 기록ㅋㅋ",
    "<b>중요</b>: 다음 시험 범위는 1-15과.",
    "문의는 mamb@snu.ac.kr (010-1234-5678)로!",
    "ㄱㄱㅋ 파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

for post in posts:
    print(clean_post(post))

print(analyze_posts(posts))

오늘 파이썬 수업 진짜 재밌었음!!
자료:
lee 팀플 어디서 모이지 #DCCP2026 #팀플 기록
중요: 다음 시험 범위는 1-15과.
문의는 [이메일] ([전화번호])로!
파이썬 진짜 좋다 #추천
{'posts_n': 6, 'avg_length_after_clean': 17.67, 'hashtag_counts': {'DCCP2026': 1, '추천': 1, '팀플': 1}, 'masked_count': 2}


clean_post의 단계 순서를 지켜야 하는 이유는 이메일이나 멘션처럼 @를 포함하는 표현을 먼저 처리하지 않으면, 자모 제거·공백 정리 등 다른 치환이 먼저 일어나면서 원래 패턴을 제대로 인식하지 못하거나 일부 문자만 남아 결과가 달라질 수 있기 때문이다.

생성형 AI https://chatgpt.com/share/6a09d09a-90ac-83ec-9e96-56d331115efe
실수로 3번 문제를 다른 대화로 풀다가 앞부분의 대화를 날려먹었습니다. 그래서 1,2번 과 3번 후반부에 대한 대화만 남아있습니다